Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Implementación de backpropagation de forma matricial**

Dataset utilizado: [California Housing](https://www.dcc.fc.up.pt/~ltorgo/Regression/cal_housing.html).

En base a 8 predictores numéricos:

- `MedInc`: median income in block group.
- `HouseAge`: median house age in block group.
- `AveRooms`: average number of rooms per household.
- `AveBedrms`: average number of bedrooms per household.
- `Population`: block group population.
- `AveOccup`: average number of household members.
- `Latitude`: block group latitude.
- `Longitude`: block group longitude.

Se busca predecir el valor de una propiedad en base a su ubicación y características.

Vamos a trabajar utilizando únicamente NumPy para las operaciones.

In [ ]:
import numpy as np
import pandas as pd

### **Arquitectura de red fija**

La arquitectura de la red será la siguiente:
* I: 8 neuronas (8 variables de entrada).
* H: 20 neuronas (única capa oculta).
* O: 1 neurona (salida).

In [ ]:
input_size = 8
hidden_size = 20
output_size = 1

Inicializamos los pesos y sesgos.

In [ ]:
np.random.seed(42)

W_input_hidden = # COMPLETAR. Sugerencia: matriz de input_size * hidden_size con valores random pequeños.
b_hidden = #COMPLETAR. Sugerencia: matriz de 1 * hidden_size con valores random pequeños.

W_hidden_output = # COMPLETAR.
b_output = # COMPLETAR.

Definimos la función de activación y su derivada. En este caso, [Leaky ReLU](https://paperswithcode.com/method/leaky-relu).

In [ ]:
def LeakyReLU(x):
    return np.where(x > 0, x, x * 0.01)

def LeakyReLU_prime(x):
    return np.where(x > 0, 1, 0.01)

Definimos la función correspondiente a la pasada *forward*.

Recibe matriz de datos de entrada `X` y devuelve `Z_hidden`, `A_hidden`, `Z_output`, `A_output`, siendo los Zs las entradas netas y las As la salidas de cada capa.

In [ ]:
def forward(X):
    global W_input_hidden, b_hidden, W_hidden_output, b_output
    
    Z_hidden = # COMPLETAR.
    A_hidden = # COMPLETAR.
    
    Z_output = # COMPLETAR.
    A_output = # COMPLETAR.
    
    return Z_hidden, Z_output, A_hidden, A_output

Definimos la función correspondiente a la pasada *backward*.

Calcular el gradiente implica obtener los deltas de cada parámetro de la red. Para ello, revisar las ecuaciones de backpropagation antes presentadas.

In [ ]:
def backward(X, Y, A1, A2, Z1, Z2):
    m = X.shape[0]
    
    delta_Z_output = (A2 - Y) * LeakyReLU_prime(Z2)
    
    delta_W_hidden_output = # COMPLETAR.
    delta_b_output = # COMPLETAR.
    
    delta_Z_hidden = # COMPLETAR.
    delta_W_input_hidden = # COMPLETAR.
    delta_b_hidden = # COMPLETAR.
    
    return delta_W_input_hidden, delta_b_hidden, delta_W_hidden_output, delta_b_output

Definimos la actualización de los parámetros.

In [ ]:
learning_rate = 0.00001

def update_parameters(delta_W_input_hidden, delta_b_hidden, delta_W_hidden_output, delta_b_output):
    global W_input_hidden, b_hidden, W_hidden_output, b_output
    
    W_input_hidden -= # COMPLETAR.
    b_hidden -= # COMPLETAR.
    
    W_hidden_output -= # COMPLETAR.
    b_output -= # COMPLETAR.

Preparamos el conjunto de datos.

In [ ]:
from sklearn.datasets import fetch_california_housing

california_data = fetch_california_housing()
X = california_data.data
Y = california_data.target
Y = Y.reshape(-1, 1)

Entrenamos.

In [ ]:
batch_size = 100
epochs = 10000
best_loss = float('inf')
best_loss_epoch = 0

for epoch in range(epochs):
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices) # Se mezclan los datos y se divide en mini batches.
    X_shuffled = X[indices]
    Y_shuffled = Y[indices]
    
    for i in range(0, X.shape[0], batch_size - 1):
        X_batch = X_shuffled[i:i+batch_size]
        Y_batch = Y_shuffled[i:i+batch_size]
        
        Z1, Z2, A1, A2 = forward(X_batch)
        delta_W_input_hidden, delta_b_hidden, delta_W_hidden_output, delta_b_output = backward(X_batch, Y_batch, A1, A2, Z1, Z2)
        update_parameters(delta_W_input_hidden, delta_b_hidden, delta_W_hidden_output, delta_b_output)
    
    Z1, Z2, A1, A2 = forward(X)
    loss = # COMPLETAR.
    
    if loss < best_loss:
        best_loss = loss
        best_loss_epoch = epoch
    
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

print(f'Best loss: {best_loss} in epoch {epoch}')

Revisemos las predicciones en el conjunto de entrenamiento.

In [ ]:
_, _, _, Y_pred = forward(X)
df = pd.DataFrame(np.concatenate([Y, Y_pred], axis = 1), columns = ['Y', 'Y_pred'])
df.head()

### **Arquitectura de red genérica**

En vez de considerar una arquitectura fija, pensemos cómo podemos generalizar el código para que funcione con
* una cantidad de capas ocultas variable y
* una cantidad de neuronas por capa también variable.

Esto va a implicar
* una cantidad de matrices de pesos W variable y
* una cantidad de vectores de sesgos b también variable.

In [ ]:
layers = [8, 16, 20, 20, 1]
# Ejemplo de posible configuración de la red, siendo 8 la entrada y 1 la salida.

Inicializamos los pesos y sesgos.

In [ ]:
np.random.seed(42)

W = []
b = []
for i in range(len(layers) - 1):
    W.append(...) # COMPLETAR.
    b.append(...) # COMPLETAR.

Definimos la función correspondiente a la pasada forward para las N capas.

In [ ]:
def forward_generic(X):
    global W
    
    Z = []
    A = [X] # Agrega X como la primera "activación" para facilitar el loop.
    
    for i in range(len(layers) - 1):
        Z_i = # COMPLETAR.
        A_i = # COMPLETAR.
        Z.append(Z_i)
        A.append(A_i)
    
    return Z, A

Definimos la función correspondiente a la pasada backward.

In [ ]:
def backward_generic(X, Y, A, Z):
    m = X.shape[0] # Cantidad de ejemplos.
    delta_Z = []
    delta_W = []
    delta_b = []
    
    # Se asume al menos una capa oculta.
    # Iniciar con la última capa.
    delta_Z_i = (A[-1] - Y) * LeakyReLU_prime(Z[-1])
    delta_Z.insert(0, delta_Z_i) # Almacenar al inicio.
    delta_W_i = 1 / m * np.matmul(A[-2].T, delta_Z_i)
    delta_W.insert(0, delta_W_i) # Almacenar al inicio.
    delta_b_i = 1 / m * np.sum(delta_Z_i, axis=0)
    delta_b.insert(0, delta_b_i) # Almacenar al inicio.
    
    # Para las capas anteriores.
    # Se itera desde la penúltima capa hasta atrás.
    # i comienza en len(layers) - 3 y termina en 0 (incluido).
    for i in range(len(layers) - 3, -1, -1): # Comienza desde la penúltima capa.
        delta_Z_i = np.matmul(...) * LeakyReLU_prime(Z[i]) # COMPLETAR.
        delta_Z.insert(0, delta_Z_i) # Almacenar al inicio.
        delta_W_i = 1 / m * np.matmul(...) # COMPLETAR.
        delta_W.insert(0, delta_W_i) # Almacenar al inicio.
        delta_b_i = 1 / m * np.sum(..., axis = 0) # COMPLETAR.
        delta_b.insert(0, delta_b_i) # Almacenar al inicio.
    
    return delta_W, delta_b

Definimos la actualización de los parámetros.

In [ ]:
learning_rate = 0.00001

def update_parameters_generic(delta_W, delta_b):
    global W, b
    for i in range(len(layers) - 1):
        W[i] -= # COMPLETAR.
        b[i] -= # COMPLETAR.

Entrenamos.

In [ ]:
batch_size = 100
epochs = 10000
best_loss = float('inf')
best_loss_epoch = 0

for epoch in range(epochs):
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)
    X_shuffled = X[indices]
    Y_shuffled = Y[indices]
    
    for i in range(0, X.shape[0], batch_size - 1):
        X_batch = X_shuffled[i:i+batch_size]
        Y_batch = Y_shuffled[i:i+batch_size]
        
        Z, A = forward_generic(X_batch)
        delta_W, delta_b = backward_generic(X_batch, Y_batch, A, Z)
        update_parameters_generic(delta_W, delta_b)
    
    Z, A = forward_generic(X)
    loss = # COMPLETAR.
    
    if loss < best_loss:
        best_loss = loss
        best_loss_epoch = epoch
    
    if epoch % 100 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item()}')

print(f'Best loss: {best_loss} in epoch {epoch}')

Revisemos las predicciones en el conjunto de entrenamiento.

In [ ]:
_, A = forward_generic(X)
df = pd.DataFrame(np.concatenate([Y, A[-1]], axis = 1), columns = ['Y', 'Y_pred'])
df.head()